# Tutorial 7: K-Nearest Neighbors and Movie Recommendation
## Find similar movies while examining sparsity, popularity, and privacy

**Course:** IE 1171  
**Files used:** `data/movies.csv` and `data/ratings.csv`  
**Level 1:** Required core—item-to-item K-nearest-neighbor recommender  
**Level 2:** Optional deep dives—sparsity, popularity filtering, distance, and privacy

---


The system does not predict a class label in the required level. Instead, it represents each movie by its pattern of user ratings and finds movies with nearby rating patterns. Claude will draft the code; you must verify the data alignment, distance definition, recommendation logic, missing-rating treatment, and societal consequences.


# <img src="tutorial-icons/learning_objectives.png" alt="Learning Objectives" width="44" style="vertical-align:middle; margin-right:10px;"> Learning Objectives

By the end of this tutorial, you should be able to:

1. **Understand nearest-neighbor recommendation**
   - Distinguish KNN classification, regression, and neighbor retrieval.
   - Represent a movie by its pattern of user ratings.
   - Explain sparsity and why a missing rating is not the same as a zero rating.
2. **Build a movie recommender**
   - Create a sparse item–user matrix and compare movies with cosine distance.
   - Map matrix positions back to the correct movie IDs and titles.
   - Return neighbors while excluding the query movie itself.
3. **Judge the recommendations**
   - Inspect distance, similarity, eligibility filters, and recommendation stability.
   - Explain how design choices can reinforce popularity feedback loops.
   - Recognize the privacy risk in detailed rating histories.


# Pólya’s Four-Step Problem-Solving Cycle

> **Backbone for this tutorial:** George Pólya’s four steps organize the work from problem framing through verification. The steps are a **cycle**, not a one-way checklist: if later evidence exposes a bad assumption, return to the earlier step that needs revision.

| Marker | Pólya step | Guiding question | In this tutorial |
|---|---|---|---|
| **🔵 🧭** | **Understand the Problem** | What is the real problem, what is known, and what constraints define success? | Define the recommendation goal and what “similar” should mean for this dataset and user task. |
| **🟣 🗺️** | **Devise a Plan** | What sequence of actions and checks should connect the current state to the goal? | Choose the item–user representation, distance rule, neighbor count, and filtering checks. |
| **🟠 🛠️** | **Carry Out the Plan** | Can the plan be executed in small, observable steps and checked as it runs? | Fit KNN and retrieve recommendations while preserving observable intermediate outputs. |
| **🟢 🔎** | **Look Back** | Does the result answer the original problem, and what should be revised or generalized? | Test recommendation quality, sensitivity, sparse-data limitations, and unintended effects. |

The colored markers reappear at the points where each step becomes the main focus. **Human Checks support the cycle, but they are not a universal checklist:** meaningful verification depends on domain knowledge, the data-generating process, and the consequences of being wrong.


# <img src="tutorial-icons/assigned_reading.png" alt="Assigned Reading" width="44" style="vertical-align:middle; margin-right:10px;"> Assigned Reading

## Statistical Reading

**James et al., _An Introduction to Statistical Learning with Applications in Python_ (ISLP)**

- Section 12.3: Missing Values and Matrix Completion
- the **“Recommender Systems”** subsection within **Section 12.3, “Missing Values and Matrix Completion”** in the attached Python edition

The book motivates recommenders through sparse user–item rating matrices and matrix completion. This tutorial then uses nearest-neighbor similarity as a related recommendation approach; it does **not** claim that ISLP Section 12.3 teaches the exact KNN recommender implementation used here.

Focus on:

- incomplete user–item matrices;
- observed and missing ratings;
- recommendation from patterns across users and items;
- the difference between storing data and producing a recommendation.

## Optional Reading

**James et al., _An Introduction to Statistical Learning with Applications in Python_ (ISLP)**

- Section 2.2: Assessing Model Accuracy
- Section 2.2.3: The Classification Setting, including the K-nearest-neighbors discussion

Level 1 and Level 2 use the same readings and the same MovieLens files. No additional dataset is introduced.

## Ethical / Social-Good Reading

**Michael Kearns and Aaron Roth, _The Ethical Algorithm_**

- Chapter 2, **“Preventing Fairness Gerrymandering”** (pp. 86–90)
- Chapter 3, **“Games People Play: The Dating Game”** (pp. 94–97)

Use these readings to ask whether recommendation quality is distributed fairly across groups and how algorithmic matching can change behavior, exposure, and future data.


# <img src="tutorial-icons/tutorial_flow.png" alt="Tutorial Flow" width="44" style="vertical-align:middle; margin-right:10px;"> Tutorial Flow

| Section | Purpose |
|---|---|
| **Theory** | Connect KNN and recommender concepts to the matrix representation. |
| **Manual Pause** | Predict what should happen before asking Claude. |
| **Claude Coding Task** | Request one focused and checkable program. |
| **Your Workspace** | Read and run Claude’s code. |
| **Reference Solution** | Compare after making your own attempt. |
| **Human Check** | Verify IDs, rows, distances, and titles. |
| **Look Back** | Examine stability, popularity, privacy, and intended use. |

> Recommendation output is not neutral discovery. The representation and filtering rules determine which items can become “nearby.”


## Tutorial Symbols

| Symbol | Meaning | What to do |
|---|---|---|
| **🔵 🧭  🟣 🗺️  🟠 🛠️  🟢 🔎** | **Pólya Backbone** | Treat the four colored checkpoints as the main problem-solving cycle; return to an earlier step when new evidence requires revision. |
| <img src="tutorial-icons/tutorial_flow.png" alt="Tutorial Flow" width="28" style="vertical-align:middle; margin-right:8px;"> | **Tutorial Flow** | Follow the notebook’s normal route. |
| <img src="tutorial-icons/learning_objectives.png" alt="Learning Objectives" width="28" style="vertical-align:middle; margin-right:8px;"> | **Learning Objectives** | See the three destinations for the tutorial. |
| <img src="tutorial-icons/assigned_reading.png" alt="Assigned Reading" width="28" style="vertical-align:middle; margin-right:8px;"> | **Assigned Reading** | Read the named sections before or alongside the notebook. |
| <img src="tutorial-icons/theory.png" alt="Theory" width="28" style="vertical-align:middle; margin-right:8px;"> | **Theory** | Connect the assigned reading to the current part. |
| <img src="tutorial-icons/manual_pause.png" alt="Manual Pause" width="28" style="vertical-align:middle; margin-right:8px;"> | **Manual Pause** | Think or predict before asking Claude. |
| <img src="tutorial-icons/without_claude.png" alt="Without Claude" width="28" style="vertical-align:middle; margin-right:8px;"> | **Without Claude** | Notice the programming details Claude can coordinate. |
| <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="28" style="vertical-align:middle; margin-right:8px;"> | **Claude Task** | Use one focused and checkable prompt. |
| <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="28" style="vertical-align:middle; margin-right:8px;"> | **Your Workspace** | Paste, read, and run Claude’s response. |
| <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="28" style="vertical-align:middle; margin-right:8px;"> | **Reference Solution** | Compare only after your own attempt. |
| <img src="tutorial-icons/human_check.png" alt="Human Check" width="28" style="vertical-align:middle; margin-right:8px;"> | **Human Check — domain expertise required** | Use the provided questions, then add a domain-specific check. The notebook cannot supply a complete checklist for every application. |
| <img src="tutorial-icons/look_back.png" alt="Look Back" width="28" style="vertical-align:middle; margin-right:8px;"> | **Look Back** | Interpret, challenge, and reflect on the result. |
| <img src="tutorial-icons/level_2.png" alt="Level 2 Challenge" width="28" style="vertical-align:middle; margin-right:8px;"> | **Level 2 — Challenge Ahead** | Take the optional harder route after Level 1. |

> **Important — Human Checks are not a complete checklist.** The notebook can suggest generic verification questions, but deciding what *must* be checked depends on knowledge of the domain, the data-generating process, and the consequences of an error. If you do not have that expertise, involve someone who does. Every Human Check asks you to add your own domain-specific question.

# <img src="tutorial-icons/theory.png" alt="Theory" width="44" style="vertical-align:middle; margin-right:10px;"> Theory Foundation: Neighbors, Distance, Sparsity, and Recommendation

K-nearest neighbors (KNN) stores the training examples and predicts from nearby cases rather than fitting one global equation. For regression,

$$
\hat f(x)=\frac{1}{K}\sum_{i\in N_K(x)}y_i,
$$

and for classification it uses a neighbor vote or class proportion. Recommendation uses the same idea: define a representation, define similarity, retrieve neighbors, and convert them into suggestions.

Distance is part of the model. Euclidean distance is

$$
d_2(x,z)=\sqrt{\sum_{j=1}^{p}(x_j-z_j)^2},
$$

so variables with large numeric scales dominate unless features are standardized. Cosine similarity,

$$
\operatorname{cos}(x,z)=\frac{x^Tz}{\lVert x\rVert_2\lVert z\rVert_2},
$$

compares direction rather than magnitude and is often useful for sparse rating vectors. It still requires a rule for missing entries and cannot tell whether non-overlap means dislike or simply no observation.

High dimension creates the **curse of dimensionality**. If each feature range is divided into $r$ intervals, the space has $r^p$ cells. To keep the same local data density as $p$ increases, the sample size must grow exponentially. Distances also tend to become less distinguishable, so the nearest case may not be meaningfully near. Irrelevant features, sparse ratings, and popularity concentration can therefore dominate a recommender.

$K$ controls a bias-variance tradeoff: small $K$ adapts locally but is sensitive to noise; large $K$ is stable but can average across genuinely different tastes. Eligibility rules, minimum overlap, weighting, and tie-breaking change the candidate set just as surely as the distance formula. Recommendation is also a feedback system: exposure affects future ratings, and those ratings affect later exposure.

### Questions you should be ready to answer

- What assumptions are hidden inside a distance measure?
- Why do scaling and missing-value rules alter the neighborhood?
- How does dimension make “nearest” less informative?
- Why is a recommendation not the same as a statement of quality?

## 🔵 🧭 Pólya Step 1 — Understand the Problem

**Backbone checkpoint.** State the real goal, evidence, constraints, and what would count as success before asking an AI system to solve anything.

**In this tutorial:** Define the recommendation goal and what “similar” should mean for this dataset and user task.

# Level 1 — Required Core

# Part 1: Define the Recommendation Problem

The supplied files contain:

## `movies.csv`

- `movieId`
- `title`

## `ratings.csv`

- `userId`
- `movieId`
- `rating`

The required task is:

> Given one movie, return movies whose user-rating patterns are most similar.

Each movie becomes a vector:

$$
X_i=(r_{i1},r_{i2},\ldots,r_{im}),
$$

where each position corresponds to one user.

The matrix is sparse because most users rate only a small fraction of all movies.

## What K Means Here

In KNN classification, $k$ often means the number of neighbors voting on a class.

In this recommender, the model performs **nearest-neighbor retrieval**. The requested number of neighbors controls how many similar movies are returned. There is no class vote in Level 1.


## <img src="tutorial-icons/theory.png" alt="Theory" width="36" style="vertical-align:middle; margin-right:9px;"> Theory: Recommendation Is Neighbor Retrieval

The statistical reading introduces KNN as a rule based on nearby observations. Here the goal is not to predict a class label; it is to retrieve movies with nearby rating patterns. The same neighbor idea is used, but the meaning of a ‘row,’ the representation, and the returned output are different.

### Deeper explanation

A neighbor system has four separate choices: representation $\phi(i)$, distance $d$, eligibility set $E$, and aggregation rule. The recommendation is $\operatorname{argmin}_{j\in E}d(\phi(q),\phi(j))$ only after those choices. Changing any one can change the result. Because there is no learned coefficient table, explanations should identify shared rating patterns and data coverage rather than claim the system discovered a stable preference trait.


## 🟣 🗺️ Pólya Step 2 — Devise a Plan

**Backbone checkpoint.** Decide the sequence of actions and checks before the main execution. Make assumptions, evaluation rules, and stopping conditions visible so they can be challenged.

**In this tutorial:** Choose the item–user representation, distance rule, neighbor count, and filtering checks.

## <img src="tutorial-icons/manual_pause.png" alt="Manual Pause" width="36" style="vertical-align:middle; margin-right:9px;"> Manual Pause: Reason About the Representation

Answer briefly:

1. What is one row of the item–user matrix?
2. What is one column?
3. What does an observed rating mean?
4. What does a blank matrix position mean?
5. Why is a blank not automatically the same as dislike?
6. Why might filling blanks with zero distort similarity?
7. Why is a sparse matrix useful?
8. What does cosine distance compare?
9. What should cosine similarity equal when cosine distance is zero?
10. Why must the query movie be removed from its own recommendation list?
11. What alignment error could return the wrong titles even when the distances are correct?


## 🟠 🛠️ Pólya Step 3 — Carry Out the Plan

**Backbone checkpoint.** Execute in small, observable steps. Read generated code or actions, stay within scope, and compare outputs with the behavior you predicted.

**In this tutorial:** Fit KNN and retrieve recommendations while preserving observable intermediate outputs.

## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 1: Load and Audit the MovieLens Files

```text
Act as a careful Python tutor.

Write one Jupyter Notebook code cell that:

1. imports pathlib and pandas;
2. loads data/movies.csv using movieId and title;
3. loads data/ratings.csv using userId, movieId, and rating;
4. uses dataframe names movies and ratings;
5. confirms that all required columns exist;
6. prints each dataframe's shape, first five rows, missing-value counts, and
   data types;
7. prints the number of unique users and unique rated movies;
8. prints the rating value counts and summary statistics;
9. checks whether every rated movieId appears in movies.csv;
10. does not merge, pivot, or fit a model.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 1


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 1


In [ ]:
from pathlib import Path
import pandas as pd

movies_path = Path("data/movies.csv")
ratings_path = Path("data/ratings.csv")

assert movies_path.exists(), f"File not found: {movies_path.resolve()}"
assert ratings_path.exists(), f"File not found: {ratings_path.resolve()}"

movies = pd.read_csv(
    movies_path,
    usecols=["movieId", "title"],
    dtype={"movieId": "int32", "title": "string"},
)

ratings = pd.read_csv(
    ratings_path,
    usecols=["userId", "movieId", "rating"],
    dtype={
        "userId": "int32",
        "movieId": "int32",
        "rating": "float32",
    },
)

print("Movies shape:", movies.shape)
display(movies.head())
print("\nMovies missing values:")
display(movies.isna().sum().to_frame("missing_count"))
print("\nMovies data types:")
display(movies.dtypes.to_frame("dtype"))

print("\nRatings shape:", ratings.shape)
display(ratings.head())
print("\nRatings missing values:")
display(ratings.isna().sum().to_frame("missing_count"))
print("\nRatings data types:")
display(ratings.dtypes.to_frame("dtype"))

print("\nUnique users:", ratings["userId"].nunique())
print("Unique rated movies:", ratings["movieId"].nunique())

print("\nRating value counts:")
display(ratings["rating"].value_counts().sort_index().to_frame("count"))

print("\nRating summary:")
display(ratings["rating"].describe().to_frame("rating"))

unknown_movie_ids = sorted(
    set(ratings["movieId"]) - set(movies["movieId"])
)
print("\nRated movie IDs missing from movies.csv:", len(unknown_movie_ids))
assert not unknown_movie_ids, (
    f"Found ratings for movie IDs absent from movies.csv: "
    f"{unknown_movie_ids[:10]}"
)


### <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


- Are movie IDs unique in `movies.csv`?
- Are there duplicate user–movie rating pairs?
- What rating scale is present?
- Are any ratings missing?
- Does every rated movie have a title?
- Do all movies have at least one rating?
- Why should title strings not be used as the matrix index when `movieId` is available?

# Part 2: Build the Item–User Matrix

This tutorial uses:

- rows = movies;
- columns = users;
- values = ratings.

A sparse matrix stores only nonzero entries, which is much more efficient than storing every empty user–movie combination.

For this introductory implementation, unobserved ratings are represented as zero before conversion to a sparse matrix. That is computationally convenient, but it is also a modeling choice. Zero means “not observed” here, not “the user rated the movie zero.”


## <img src="tutorial-icons/theory.png" alt="Theory" width="36" style="vertical-align:middle; margin-right:9px;"> Theory: Missing Ratings Create a Sparse Representation

A user–movie matrix is mostly unobserved because most users rate only a small fraction of movies. Sparse storage keeps the observed entries efficiently. Filling an unobserved rating with zero is a computational representation, not evidence that the user disliked the movie, so that choice must be remembered when interpreting distance.

### Deeper explanation

In a user–item matrix $R\in\mathbb{R}^{m\times n}$, the observed set $\Omega$ is usually much smaller than $mn$. Treating every missing entry as zero confounds “not rated” with a true zero rating. Pairwise similarity can use co-rated items, but small overlap makes estimates unstable. Minimum-overlap rules reduce noise at the cost of excluding long-tail users or items.


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 2: Create and Verify the Sparse Matrix

```text
Using movies and ratings:

1. check for duplicate userId-movieId pairs and print their count;
2. if duplicates exist, average them with pivot_table;
3. create an item-user dataframe with movieId as rows, userId as columns,
   and rating as values;
4. fill missing positions with 0 only for matrix construction;
5. convert the values to scipy.sparse.csr_matrix;
6. name the dataframe movie_user_ratings;
7. name the sparse matrix movie_user_matrix;
8. create movie_ids_in_matrix in exact row order;
9. create movie_id_to_row mapping each movieId to its matrix-row position;
10. print the matrix shape, number of nonzero entries, and sparsity percentage;
11. assert that the mapping and row order agree.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 2


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 2


In [ ]:
import numpy as np
from scipy.sparse import csr_matrix

duplicate_pairs = ratings.duplicated(
    subset=["userId", "movieId"],
    keep=False,
).sum()

print("Rows belonging to duplicate user-movie pairs:", duplicate_pairs)

movie_user_ratings = ratings.pivot_table(
    index="movieId",
    columns="userId",
    values="rating",
    aggfunc="mean",
    fill_value=0,
)

movie_user_matrix = csr_matrix(movie_user_ratings.to_numpy())
movie_ids_in_matrix = movie_user_ratings.index.to_numpy()

movie_id_to_row = {
    int(movie_id): row_position
    for row_position, movie_id in enumerate(movie_ids_in_matrix)
}

total_positions = (
    movie_user_matrix.shape[0] * movie_user_matrix.shape[1]
)
nonzero_positions = movie_user_matrix.nnz
sparsity_percent = (
    100 * (1 - nonzero_positions / total_positions)
    if total_positions
    else float("nan")
)

print("Matrix shape:", movie_user_matrix.shape)
print("Nonzero ratings:", nonzero_positions)
print(f"Sparsity: {sparsity_percent:.2f}%")

assert len(movie_ids_in_matrix) == movie_user_matrix.shape[0]
assert len(movie_id_to_row) == movie_user_matrix.shape[0]

for row_position, movie_id in enumerate(movie_ids_in_matrix[:10]):
    assert movie_id_to_row[int(movie_id)] == row_position


### <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


1. How many matrix positions exist?
2. How many contain observed ratings?
3. What percentage is unobserved?
4. Did the matrix preserve `movieId` row order?
5. Why is `movie_id_to_row` necessary?
6. What error would occur if a neighbor row number were used directly as a row number in `movies.csv`?
7. What does zero mean in the constructed matrix?
8. Why is that interpretation a limitation?

# Part 3: Fit the Nearest-Neighbor Model

This tutorial uses:

```python
NearestNeighbors(metric="cosine", algorithm="brute")
```

Cosine similarity compares the angle between two rating vectors.

If cosine distance is $d$, then:

$$
\text{cosine similarity}=1-d.
$$

Smaller distance means more similar rating patterns.


## <img src="tutorial-icons/theory.png" alt="Theory" width="36" style="vertical-align:middle; margin-right:9px;"> Theory: Cosine Distance Compares Rating-Pattern Direction

Cosine similarity is high when two movie vectors point in similar directions, even if their total numbers or magnitudes of ratings differ. Cosine distance is `1 - similarity`; smaller values indicate closer represented patterns. It measures similarity in the constructed matrix, not similarity of plot, genre, or artistic quality by itself.

### Deeper explanation

Cosine similarity is invariant to multiplying an entire vector by a positive constant, so it emphasizes pattern direction. Mean-centering changes the question from similar raw ratings to similar deviations from each user's average. With sparse vectors, shared zeros can be misleading and undefined zero-norm vectors require explicit handling. Always state the vector construction before interpreting the similarity.


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 3: Fit the Cosine Neighbor Model

```text
Using movie_user_matrix:

1. import sklearn.neighbors.NearestNeighbors;
2. create a model using metric="cosine" and algorithm="brute";
3. fit the model on movie_user_matrix;
4. name the fitted model movie_knn;
5. print the number of movie rows and user columns;
6. do not request recommendations yet.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 3


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 3


In [ ]:
from sklearn.neighbors import NearestNeighbors

movie_knn = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
)

movie_knn.fit(movie_user_matrix)

print("Movie rows fitted:", movie_user_matrix.shape[0])
print("User columns used:", movie_user_matrix.shape[1])


### <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


- Was the model fit on movie rows rather than user rows?
- Is the metric cosine?
- Does fitting the model create new ratings?
- Does KNN learn regression coefficients?
- What information does the model store?
- Why is this often called a “lazy” learning method?

# Part 4: Search Titles and Return Recommendations

A reliable recommendation function must coordinate four spaces:

1. title text;
2. `movieId`;
3. sparse-matrix row position;
4. neighbor distance.

A polished list can still be wrong if any mapping is misaligned.


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 4: Write Search and Recommendation Functions

```text
Using movies, movie_knn, movie_user_matrix, movie_id_to_row, and
movie_ids_in_matrix, write Python code that:

1. creates a title lookup from movieId to title;
2. defines search_titles(query, limit=10) using case-insensitive substring
   matching and returning movieId and title;
3. defines recommend_by_title(exact_title, n_recommendations=10);
4. requires an exact case-insensitive full-title match after search;
5. raises a clear error if no exact title is found;
6. raises a clear error if the movie has no ratings row;
7. requests n_recommendations + 1 neighbors so the query movie can be removed;
8. converts each neighbor row back to the correct movieId and title;
9. excludes the query movie;
10. returns a dataframe with Rank, movieId, Title, CosineDistance, and
    CosineSimilarity;
11. handles the case where fewer neighbors are available.

Do not use fuzzywuzzy or another dataset.
Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 4


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 4


In [ ]:
title_by_movie_id = (
    movies.drop_duplicates("movieId")
    .set_index("movieId")["title"]
    .to_dict()
)

def search_titles(query, limit=10):
    # Return case-insensitive substring matches from movies.csv.
    query_text = str(query).strip()

    if not query_text:
        raise ValueError("Enter at least one character to search.")

    matches = movies.loc[
        movies["title"].str.contains(
            query_text,
            case=False,
            na=False,
            regex=False,
        ),
        ["movieId", "title"],
    ].drop_duplicates()

    return matches.head(limit).reset_index(drop=True)


def recommend_by_title(exact_title, n_recommendations=10):
    # Return nearest movies for one exact full title.
    if n_recommendations < 1:
        raise ValueError("n_recommendations must be at least 1.")

    exact_matches = movies.loc[
        movies["title"].str.casefold()
        == str(exact_title).strip().casefold(),
        ["movieId", "title"],
    ].drop_duplicates()

    if exact_matches.empty:
        suggestions = search_titles(exact_title, limit=10)
        raise ValueError(
            "No exact title match was found. "
            f"Search suggestions:\n{suggestions.to_string(index=False)}"
        )

    query_movie_id = int(exact_matches.iloc[0]["movieId"])

    if query_movie_id not in movie_id_to_row:
        raise ValueError(
            f"{exact_title!r} exists in movies.csv but has no ratings row."
        )

    query_row = movie_id_to_row[query_movie_id]
    available_neighbors = movie_user_matrix.shape[0]

    requested_neighbors = min(
        n_recommendations + 1,
        available_neighbors,
    )

    distances, neighbor_rows = movie_knn.kneighbors(
        movie_user_matrix[query_row],
        n_neighbors=requested_neighbors,
    )

    recommendation_rows = []

    for distance, neighbor_row in zip(
        distances.ravel(),
        neighbor_rows.ravel(),
    ):
        neighbor_movie_id = int(movie_ids_in_matrix[neighbor_row])

        if neighbor_movie_id == query_movie_id:
            continue

        recommendation_rows.append(
            {
                "movieId": neighbor_movie_id,
                "Title": title_by_movie_id.get(
                    neighbor_movie_id,
                    "<title missing>",
                ),
                "CosineDistance": float(distance),
                "CosineSimilarity": float(1 - distance),
            }
        )

        if len(recommendation_rows) == n_recommendations:
            break

    recommendations = pd.DataFrame(recommendation_rows)
    recommendations.insert(
        0,
        "Rank",
        np.arange(1, len(recommendations) + 1),
    )

    return recommendations


### <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


Trace one recommendation completely:

1. What exact title was selected?
2. Which `movieId` belongs to it?
3. Which sparse-matrix row belongs to that ID?
4. Which row did KNN return as the nearest non-self neighbor?
5. Which `movieId` belongs to the neighbor row?
6. Which title belongs to that ID?
7. Does similarity equal `1 - distance`?
8. Is the query movie absent from the returned list?

This trace is more important than whether the titles “feel reasonable.”

# Part 5: Test the Recommender

The recommender is tested with a query related to “Jurassic Park.”

Because MovieLens titles often include release years, first search and then use the exact returned title.


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 5: Search and Recommend

```text
Using search_titles and recommend_by_title:

1. search for "Jurassic Park";
2. display up to 10 matching titles;
3. select one exact title from the displayed course data;
4. request 10 recommendations;
5. display the recommendation table;
6. print the selected title;
7. do not hard-code a title that was not found in the search result.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 5


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 5


In [ ]:
jurassic_matches = search_titles("Jurassic Park", limit=10)
display(jurassic_matches)

if jurassic_matches.empty:
    raise ValueError("No title containing 'Jurassic Park' was found.")

selected_title = jurassic_matches.iloc[0]["title"]
print("Selected title:", selected_title)

jurassic_recommendations = recommend_by_title(
    selected_title,
    n_recommendations=10,
)

display(jurassic_recommendations)


## 🟢 🔎 Pólya Step 4 — Look Back

**Backbone checkpoint.** Do not stop at “it ran.” Ask whether the result answers the original problem, what evidence supports it, what failed, and what should change. Domain expertise matters here because a generic checklist cannot know every real-world failure mode.

**In this tutorial:** Test recommendation quality, sensitivity, sparse-data limitations, and unintended effects.

### <img src="tutorial-icons/look_back.png" alt="Look Back" width="30" style="vertical-align:middle; margin-right:8px;"> Look Back

For the returned list:

1. Are the distances sorted from smallest to largest?
2. Are similarities sorted from largest to smallest?
3. Are all titles different from the query title?
4. Do any recommendations have surprisingly little rating support?
5. Could two movies appear similar because they share only a few raters?
6. Are the results based on plot, genre, actors, or user-rating patterns?
7. Why should you not invent a narrative explanation that the data did not test?
8. Would the same recommendations appear after the ratings data changed?


# AI for Social Good: Recommendation Systems Shape Choice, Visibility, and Feedback

KNN and recommendation connect to two broader social-good themes: **algorithmic accountability** and the way social influence can change what becomes popular. A recommender does not merely discover preference; it also changes future exposure, ratings, and behavior.

Recommendation systems can help people navigate large collections and discover useful material. They can also:

- reinforce popularity and reduce exposure to less-visible items;
- narrow a user’s information environment;
- create feedback loops in which what is shown becomes what is later observed as “preferred”;
- perform well overall while serving some groups poorly;
- infer sensitive preferences from behavioral traces;
- expose users when supposedly anonymous histories can be linked to outside information.

Before deploying a recommender, ask:

1. What is being optimized: relevance, engagement, diversity, well-being, or something else?
2. Who receives less visibility because of the recommendation rule?
3. Are there small or intersecting groups whose experience is hidden by average metrics?
4. What data would users reasonably expect to be used for recommendation?
5. Can users understand, change, or escape the recommendations?

> **Social-good principle:** A recommender shapes the environment it later learns from. Evaluation must include influence, privacy, and who gets seen—not only similarity scores.


# Tutorial 7 Conclusion

You used Claude to:

1. load the original MovieLens files;
2. inspect ratings and identifiers;
3. construct a sparse item–user matrix;
4. preserve the mapping between movie IDs and matrix rows;
5. fit a cosine nearest-neighbor model;
6. search titles safely;
7. return and verify similar movies;
8. interpret distance and similarity;
9. connect recommendations to sparsity, feedback, and privacy.

The central technical lesson is that neighbor calculations are only as trustworthy as the representation and ID alignment beneath them.


# <img src="tutorial-icons/look_back.png" alt="Look Back" width="44" style="vertical-align:middle; margin-right:10px;"> Final Reflection

Answer briefly:

1. What does one row of the matrix represent?
2. Why is the matrix sparse?
3. What does zero mean in the constructed matrix?
4. Why is that meaning imperfect?
5. What does cosine distance measure?
6. How is this recommender different from KNN classification?
7. Why must row positions be mapped back to `movieId`?
8. Which assertion or trace best protects against a title-alignment error?
9. What did Claude make easier?
10. Which interpretation remained a human responsibility?
11. How can recommendation create a feedback loop?
12. Why can anonymized ratings still threaten privacy?


# <img src="tutorial-icons/level_2.png" alt="Level 2 Challenge" width="44" style="vertical-align:middle; margin-right:10px;"> Level 2 — Optional Deep Dives

> **Challenge ahead:** Complete Level 1 first; this optional route adds a harder application of the same reading.

# Part 6: Minimum-Rating Filters and Sparse Neighbors

A movie with very few ratings can appear artificially similar to another movie because the comparison contains little shared evidence.

A minimum-rating filter asks:

> How many observed ratings must a movie have before it is eligible to appear in the neighbor matrix?

This does not solve every sparsity problem. It makes one eligibility choice explicit.


## <img src="tutorial-icons/theory.png" alt="Theory" width="36" style="vertical-align:middle; margin-right:9px;"> Theory: Eligibility Rules Change the Candidate Set

A minimum-rating filter does not change the neighbor formula. It changes which movies are allowed to compete for a recommendation. Raising the minimum can make similarities more stable while excluding newer or less popular items, so the filter is both a statistical and a visibility decision.

### Deeper explanation

Filtering by minimum rating count, recency, genre, or availability changes who can be recommended. This is a form of policy, not preprocessing trivia. A popularity threshold reduces variance but can create exposure inequality: already-visible items collect more data and remain eligible. Report both relevance measures and coverage—the fraction of users or items for which the system can make recommendations.


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 6: Compare No Filter With a 20-Rating Filter

```text
Using the same ratings and movies data:

1. count ratings per movieId;
2. identify movie IDs with at least 20 ratings;
3. rebuild a filtered item-user matrix using only those movie IDs;
4. fit a cosine NearestNeighbors model on the filtered matrix;
5. create the row mappings needed for correct title lookup;
6. write a compact helper that returns recommendations from the filtered model;
7. use the same selected Jurassic Park title if it remains eligible;
8. compare the original and filtered top-10 lists in one table;
9. calculate how many titles overlap;
10. do not load a new dataset.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 6


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 6


In [ ]:
rating_count_by_movie = ratings.groupby("movieId").size()
eligible_movie_ids = rating_count_by_movie.loc[
    rating_count_by_movie >= 20
].index

filtered_ratings = ratings.loc[
    ratings["movieId"].isin(eligible_movie_ids)
].copy()

filtered_movie_user_ratings = filtered_ratings.pivot_table(
    index="movieId",
    columns="userId",
    values="rating",
    aggfunc="mean",
    fill_value=0,
)

filtered_movie_user_matrix = csr_matrix(
    filtered_movie_user_ratings.to_numpy()
)
filtered_movie_ids = filtered_movie_user_ratings.index.to_numpy()
filtered_movie_id_to_row = {
    int(movie_id): row_position
    for row_position, movie_id in enumerate(filtered_movie_ids)
}

filtered_knn = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
)
filtered_knn.fit(filtered_movie_user_matrix)


def recommend_from_filtered_model(
    exact_title,
    n_recommendations=10,
):
    exact_matches = movies.loc[
        movies["title"].str.casefold()
        == str(exact_title).strip().casefold(),
        ["movieId", "title"],
    ].drop_duplicates()

    if exact_matches.empty:
        raise ValueError("No exact title match was found.")

    query_movie_id = int(exact_matches.iloc[0]["movieId"])

    if query_movie_id not in filtered_movie_id_to_row:
        raise ValueError(
            f"{exact_title!r} has fewer than 20 ratings "
            "and is not eligible for this filtered model."
        )

    query_row = filtered_movie_id_to_row[query_movie_id]
    requested_neighbors = min(
        n_recommendations + 1,
        filtered_movie_user_matrix.shape[0],
    )

    distances, rows = filtered_knn.kneighbors(
        filtered_movie_user_matrix[query_row],
        n_neighbors=requested_neighbors,
    )

    output_rows = []

    for distance, row_position in zip(
        distances.ravel(),
        rows.ravel(),
    ):
        neighbor_movie_id = int(filtered_movie_ids[row_position])

        if neighbor_movie_id == query_movie_id:
            continue

        output_rows.append(
            {
                "Title": title_by_movie_id.get(
                    neighbor_movie_id,
                    "<title missing>",
                ),
                "FilteredDistance": float(distance),
                "FilteredSimilarity": float(1 - distance),
                "RatingCount": int(
                    rating_count_by_movie.loc[neighbor_movie_id]
                ),
            }
        )

        if len(output_rows) == n_recommendations:
            break

    return pd.DataFrame(output_rows)


original_top10 = recommend_by_title(
    selected_title,
    n_recommendations=10,
)
filtered_top10 = recommend_from_filtered_model(
    selected_title,
    n_recommendations=10,
)

comparison = pd.DataFrame(
    {
        "OriginalTitle": original_top10["Title"],
        "FilteredTitle": filtered_top10["Title"],
    }
)

overlap_count = len(
    set(original_top10["Title"])
    & set(filtered_top10["Title"])
)

print("Selected title:", selected_title)
print("Movies eligible with at least 20 ratings:", len(eligible_movie_ids))
print("Top-10 title overlap:", overlap_count)
display(comparison)
display(filtered_top10)


### Filter Reflection

1. How many movies were removed by the eligibility rule?
2. How much did the top-10 list change?
3. Did the filtered recommendations have more rating support?
4. Which niche movies can no longer be recommended?
5. Does the filter improve reliability, reduce diversity, or both?
6. Who benefits from requiring popularity before eligibility?
7. Why is `20` a modeling choice rather than a universal rule?


# Part 7: Distance Choice and Recommendation Stability

KNN can use different distance functions.

The baseline recommender uses cosine distance because it compares rating-pattern direction. Euclidean distance instead emphasizes absolute coordinate differences.

The two metrics answer different similarity questions.


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 7: Compare Cosine and Euclidean Neighbors

```text
Using the 20-rating filtered matrix:

1. fit one NearestNeighbors model with cosine distance;
2. fit one with euclidean distance;
3. query the same selected movie;
4. return the top 10 non-self titles for each metric;
5. include each metric's distance;
6. calculate title overlap between the two lists;
7. display a side-by-side comparison;
8. do not declare one metric best without an evaluation objective.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 7


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 7


In [ ]:
cosine_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
).fit(filtered_movie_user_matrix)

euclidean_model = NearestNeighbors(
    metric="euclidean",
    algorithm="brute",
).fit(filtered_movie_user_matrix)


def neighbor_table_for_metric(model, distance_name, top_n=10):
    query_movie_id = int(
        movies.loc[
            movies["title"].str.casefold()
            == str(selected_title).casefold(),
            "movieId",
        ].iloc[0]
    )

    query_row = filtered_movie_id_to_row[query_movie_id]
    n_neighbors = min(top_n + 1, filtered_movie_user_matrix.shape[0])

    distances, rows = model.kneighbors(
        filtered_movie_user_matrix[query_row],
        n_neighbors=n_neighbors,
    )

    results = []

    for distance, row_position in zip(
        distances.ravel(),
        rows.ravel(),
    ):
        movie_id = int(filtered_movie_ids[row_position])

        if movie_id == query_movie_id:
            continue

        results.append(
            {
                "Title": title_by_movie_id.get(movie_id, "<title missing>"),
                distance_name: float(distance),
            }
        )

        if len(results) == top_n:
            break

    return pd.DataFrame(results)


cosine_neighbors = neighbor_table_for_metric(
    cosine_model,
    "CosineDistance",
)
euclidean_neighbors = neighbor_table_for_metric(
    euclidean_model,
    "EuclideanDistance",
)

metric_comparison = pd.DataFrame(
    {
        "CosineTitle": cosine_neighbors["Title"],
        "CosineDistance": cosine_neighbors["CosineDistance"],
        "EuclideanTitle": euclidean_neighbors["Title"],
        "EuclideanDistance": euclidean_neighbors["EuclideanDistance"],
    }
)

metric_overlap = len(
    set(cosine_neighbors["Title"])
    & set(euclidean_neighbors["Title"])
)

print("Top-10 overlap between metrics:", metric_overlap)
display(metric_comparison)


### Distance Reflection

1. Which titles appear under both metrics?
2. Which titles change?
3. Does a smaller numeric distance mean the same thing across metrics?
4. Why should raw cosine and Euclidean distance values not be compared directly?
5. Which metric better matches the question “rated in a similar pattern”?
6. What held-out evaluation would be needed before calling one metric better?
7. Why is recommendation quality not captured by whether a few titles seem familiar?


# Part 8: Use Claude as a Critical Recommendation Partner

Copy this prompt into Claude:

```text
Act as a skeptical reviewer of the MovieLens nearest-neighbor recommender.

Use only the current readings, movies.csv, ratings.csv, and the notebook outputs.

1. explain how missing ratings were represented;
2. identify one way that representation can distort similarity;
3. explain how a minimum-rating filter changes eligibility;
4. explain how cosine and Euclidean distance answer different questions;
5. identify one popularity feedback loop;
6. connect rating histories to the assigned privacy reading;
7. identify one technical validation the notebook has completed;
8. identify one important recommendation-quality validation it has not completed;
9. propose one next check using only the same MovieLens files;
10. ask me one question that tests whether I understand the difference between
    a plausible recommendation and a validated recommender.

Do not suggest another algorithm, a new reading, or a new dataset.
```

The Level 2 goal is to deepen the current method, not to collect additional algorithms.


# Sources and Course Resources

- James, Gareth, Daniela Witten, Trevor Hastie, and Robert Tibshirani. _An Introduction to Statistical Learning_, Sections 12.3 and 2.2.
- MovieLens dataset source: [GroupLens](https://grouplens.org/datasets/movielens/).
- Salganik, Matthew J., Peter Sheridan Dodds, and Duncan J. Watts. “Experimental Study of Inequality and Unpredictability in an Artificial Cultural Market.” _Science_ 311, no. 5762 (2006): 854–856. [Music Lab project](http://www.princeton.edu/~mjs3/musiclab.shtml).
- Narayanan, Arvind, and Vitaly Shmatikov. “How to Break Anonymity of the Netflix Prize Dataset.” arXiv:cs/0610105.
- Additional background resource: [Netflix Prize](https://en.wikipedia.org/wiki/Netflix_Prize).
